# Lesson 8: LangGraph Fundamentals — Inside the Black Box

## WHY

In the last six lessons, you built a moderation agent using `create_agent()`. Every time you
called it, you got back a `CompiledStateGraph` — then you called `.invoke()` or `.ainvoke()` on it.

That black box *is* LangGraph's `StateGraph`. This lesson opens it.

**Why does this matter?**

1. **`create_agent` is a shortcut** for one specific graph pattern (the ReAct loop). Your
   moderation pipeline needs a *different* pattern: sequential nodes with conditional routing.
   `create_agent` cannot express that — you need `StateGraph` directly.

2. **Debugging** — when something fails, knowing *which node* failed and *what state it had*
   turns a 2-hour session into a 5-minute fix.

3. **The `recursion_limit` finally makes sense** — it counts supersteps (graph iterations), not
   LLM calls. Understanding graph structure tells you exactly what number to use.

4. **Foundation for Lessons 9–17** — memory, streaming, multi-agent patterns, and checkpointing
   all work at the graph level. This lesson is the prerequisite for all of them.

**By the end of this lesson you will:**
1. Define a custom `TypedDict` state schema with message reducers
2. Write node functions that read from state and return partial updates
3. Wire nodes with unconditional and conditional edges
4. Visualize any graph with Mermaid
5. Build a 3-node moderation pipeline from scratch
6. Add real LLM calls to each node
7. Compare your manual graph against what `create_agent` generates
8. Package everything as a reusable factory function

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print('API key loaded:', 'OPENAI_API_KEY' in os.environ)

## WHAT — The Four Primitives

LangGraph builds any agentic workflow from four concepts:

| Primitive | What it is | Python |
|-----------|-----------|--------|
| **State** | Snapshot of everything your graph knows | `TypedDict` class |
| **Node** | A unit of work that transforms state | Regular `def` function |
| **Edge** | A connection between nodes | `.add_edge()` / `.add_conditional_edges()` |
| **Graph** | The orchestrator | `StateGraph` → `.compile()` → `CompiledStateGraph` |

**Data flow:**
1. You call `graph.invoke(initial_state)`
2. The graph routes to the entry node
3. Each node reads the full state, does its work, returns a *partial* update
4. Updates are merged back into state using reducers
5. The graph follows edges to the next node
6. Repeat until `END`

`create_agent()` uses this exact same machinery — it just hides the graph-building step from you.

## WHAT — State: The Shared Memory of Your Graph

State is a `TypedDict` — Python's typed dictionary class. Every node reads from and writes to it.

### Reducers: how list fields get merged

For plain fields (`str`, `int`, `bool`), the last write wins. For lists — especially message
history — you usually want to *append* instead of replace. That's what a **reducer** does:

```python
from typing import Annotated
from langgraph.graph.message import add_messages

# Without reducer: each node update replaces the list
messages: list                           # last write wins

# With add_messages reducer: new messages are appended
messages: Annotated[list, add_messages]  # accumulates all messages
```

### `MessagesState` — the built-in shorthand

LangGraph 1.1 ships a pre-built state for message-based apps:

```python
from langgraph.graph import MessagesState

# MessagesState is exactly equivalent to:
# class MessagesState(TypedDict):
#     messages: Annotated[list[BaseMessage], add_messages]
```

You can subclass it to add extra fields. Or define everything from scratch with `TypedDict`.
Both approaches work in LangGraph 1.1 — this lesson uses explicit `TypedDict` for clarity.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages
from langgraph.graph import MessagesState


# ── Approach 1: Subclass MessagesState (compact) ──────────────────────────────

class CompactModerationState(MessagesState):
    # messages field inherited with add_messages reducer
    verdict: str
    reason: str
    confidence: float
    action_taken: str


# ── Approach 2: Explicit TypedDict (used throughout this lesson) ──────────────

class ModerationState(TypedDict):
    messages: Annotated[list, add_messages]   # reducer: appends, never replaces
    verdict: str        # 'clean' | 'flagged'
    reason: str         # explanation from triage LLM
    confidence: float   # 0.0 to 1.0 certainty score
    action_taken: str   # what the moderator decided


print('CompactModerationState fields:', list(CompactModerationState.__annotations__.keys()))
print('ModerationState fields:       ', list(ModerationState.__annotations__.keys()))

## WHAT — Nodes: Functions That Transform State

A node is a plain Python function. The only contract:

```python
def my_node(state: MyState) -> dict:
    # Read from full state
    value = state['some_field']

    # Do work (call LLM, run business logic, query a database...)
    result = do_something(value)

    # Return ONLY the fields you changed — not the whole state
    return {'some_field': result}
```

**What you return is a diff**, not a replacement. Fields you don't include stay unchanged.

For `messages` fields using `add_messages`, returning `{'messages': [new_msg]}` *appends*
`new_msg` to the list — it does not replace it. That is the reducer at work.

`async def` nodes also work — required when calling Discord APIs or anything that uses `await`.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage


# ── Stub node 1: triage ───────────────────────────────────────────────────────

def triage_node_stub(state: ModerationState) -> dict:
    # Stub: always returns 'clean'. Real version calls an LLM.
    message_content = state['messages'][-1].content if state['messages'] else ''
    print(f'  [triage] checking: {message_content!r}')
    return {
        'verdict': 'clean',
        'reason': 'stub — always passes',
        'confidence': 1.0,
    }


# ── Stub node 2: passthrough (no LLM needed even in production) ───────────────

def passthrough_node(state: ModerationState) -> dict:
    # Message was clean — no action required.
    print('  [passthrough] message is clean, no action needed')
    return {'action_taken': 'none'}


# ── Stub node 3: moderate ─────────────────────────────────────────────────────

def moderate_node_stub(state: ModerationState) -> dict:
    # Stub: always deletes flagged messages.
    reason = state['reason']
    print(f'  [moderate] taking action because: {reason}')
    return {'action_taken': f'deleted (reason: {reason})'}


print('Three stub nodes defined.')
print('Each reads from state and returns a partial update dict.')

## WHAT — Edges: Routing Between Nodes

### Unconditional edges — always go from A to B

```python
from langgraph.graph import START, END

builder.add_edge(START, 'entry_node')   # START -> entry_node always
builder.add_edge('node_a', 'node_b')   # A -> B always
builder.add_edge('last_node', END)     # terminates execution
```

### Conditional edges — the router function decides

```python
def route_verdict(state: ModerationState) -> Literal['moderate', 'passthrough']:
    return 'moderate' if state['verdict'] == 'flagged' else 'passthrough'

builder.add_conditional_edges('triage', route_verdict)
# LangGraph reads the Literal type hint to know the possible destinations.
# No need to pass a mapping dict — it is inferred automatically.
```

The router receives the full state and returns a **string matching an existing node name** (or
`END`). The `Literal` type hint lets LangGraph validate destinations at compile time.

### Convenience: `add_sequence`

LangGraph 1.1 adds `add_sequence([node_a, node_b, node_c])` as a shortcut for chaining nodes
in a strict linear order. It calls `add_node` and wires the edges for you.

### Legacy aliases (LangGraph < 1.0)

Older code uses `set_entry_point()` and `set_finish_point()` — these still work in 1.1 but
`add_edge(START, ...)` is preferred for consistency.

## HOW — Step 1: Build the Stub Moderation Graph

Now let's wire the four primitives together. Stub nodes first — focus on structure, not LLM calls.

**Target graph:**

```
START -> triage --[flagged]--> moderate    -> END
                --[clean]---> passthrough  -> END
```

Three nodes. One conditional branch. Two terminal edges.

In [ ]:
from langgraph.graph import StateGraph, START, END


# ── 1. Create the builder ─────────────────────────────────────────────────────

builder = StateGraph(ModerationState)


# ── 2. Add nodes ──────────────────────────────────────────────────────────────

builder.add_node('triage', triage_node_stub)
builder.add_node('moderate', moderate_node_stub)
builder.add_node('passthrough', passthrough_node)


# ── 3. Define the conditional router ─────────────────────────────────────────

def route_verdict(state: ModerationState) -> Literal['moderate', 'passthrough']:
    return 'moderate' if state['verdict'] == 'flagged' else 'passthrough'


# ── 4. Add edges ──────────────────────────────────────────────────────────────

builder.add_edge(START, 'triage')                         # entry point
builder.add_conditional_edges('triage', route_verdict)   # conditional branch
builder.add_edge('moderate', END)                        # terminal
builder.add_edge('passthrough', END)                     # terminal


# ── 5. Compile ────────────────────────────────────────────────────────────────

stub_graph = builder.compile()

print('Graph compiled successfully!')
print('Type:', type(stub_graph))
print()
print('The same CompiledStateGraph type that create_agent() returns.')

In [ ]:
# draw_mermaid() returns the Mermaid diagram as a string.
# Paste it at https://mermaid.live to see a rendered flowchart.
print(stub_graph.get_graph().draw_mermaid())

In [ ]:
from langchain_core.messages import HumanMessage


initial_state = {
    'messages': [HumanMessage(content='Hello everyone, hope you had a great weekend!')],
    'verdict': '',
    'reason': '',
    'confidence': 0.0,
    'action_taken': '',
}

print('Invoking with a clean message...')
print()

result = stub_graph.invoke(initial_state)

print()
print('Final state:')
print(f'  verdict:      {result["verdict"]}')
print(f'  reason:       {result["reason"]}')
print(f'  confidence:   {result["confidence"]}')
print(f'  action_taken: {result["action_taken"]}')

In [ ]:
import pprint

pprint.pprint(result)

In [ ]:
# stream() yields one event per node execution.
# Each event is {node_name: partial_state_update}.
# This is how you observe what each node does step by step.
print('Streaming: watch state update per node')
print()

for event in stub_graph.stream(initial_state):
    for node_name, node_output in event.items():
        print(f'[{node_name}] returned:')
        for key, value in node_output.items():
            if key != 'messages':  # skip the message list for brevity
                print(f'  {key}: {value!r}')
        print()

## HOW — Step 2: Add Real LLM Nodes

The stub graph proved the structure works. Now replace stub nodes with real LLM calls.

Each node will use `ChatOpenAI.with_structured_output()` to get a typed Pydantic response — the
same pattern from Lesson 5.

**Node plan:**
- `triage_node` — fast classification: is this message a policy violation?
- `moderate_node` — action decision: what should we do about it?
- `passthrough_node` — no-op for clean messages (no LLM needed)

> **Cost tip:** `gpt-4o-mini` for both nodes. Triage is classification, not reasoning —
> you do not need GPT-4.

In [ ]:
from pydantic import BaseModel, Field


class TriageResult(BaseModel):
    verdict: Literal['clean', 'flagged']
    reason: str = Field(description='Brief explanation of the decision')
    confidence: float = Field(ge=0.0, le=1.0, description='Certainty score, 0 to 1')


class ModerationDecision(BaseModel):
    action: Literal['delete', 'warn', 'timeout', 'none']
    explanation: str = Field(description='Why this action was chosen')


print('TriageResult fields:', list(TriageResult.model_fields.keys()))
print('ModerationDecision fields:', list(ModerationDecision.model_fields.keys()))

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage


llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
triage_llm = llm.with_structured_output(TriageResult)

TRIAGE_SYSTEM_PROMPT = (
    'You are a Discord moderation assistant. '
    'Analyze the message for policy violations: spam, hate speech, '
    'harassment, NSFW content, or unsolicited self-promotion. '
    'Respond with a structured verdict.'
)


def triage_node(state: ModerationState) -> dict:
    message_content = state['messages'][-1].content if state['messages'] else ''
    result: TriageResult = triage_llm.invoke([
        SystemMessage(content=TRIAGE_SYSTEM_PROMPT),
        HumanMessage(content=f'Message to analyze: {message_content!r}'),
    ])
    return {
        'verdict': result.verdict,
        'reason': result.reason,
        'confidence': result.confidence,
    }


print('triage_node defined — calls gpt-4o-mini with structured output')

In [ ]:
MODERATION_SYSTEM_PROMPT = (
    'You are a Discord moderator deciding the right action for a flagged message. '
    'Choose the minimum necessary intervention: '
    'delete for clear violations (spam, slurs, NSFW), '
    'warn for borderline content or apparent first offenses, '
    'timeout for repeated violations or serious harassment, '
    'none if on reflection the message does not warrant action.'
)

moderate_llm = llm.with_structured_output(ModerationDecision)


def moderate_node(state: ModerationState) -> dict:
    msg = state['messages'][-1].content if state['messages'] else ''
    reason = state['reason']
    conf = state['confidence']
    user_content = f'Flagged message: {msg!r} | Triage reason: {reason} | Confidence: {conf:.0%}'
    result: ModerationDecision = moderate_llm.invoke([
        SystemMessage(content=MODERATION_SYSTEM_PROMPT),
        HumanMessage(content=user_content),
    ])
    return {'action_taken': f'{result.action}: {result.explanation}'}


print('moderate_node defined — calls gpt-4o-mini to decide action')

In [ ]:
# Build the full graph — same structure, real LLM nodes instead of stubs.
real_builder = StateGraph(ModerationState)
real_builder.add_node('triage', triage_node)
real_builder.add_node('moderate', moderate_node)
real_builder.add_node('passthrough', passthrough_node)  # still a no-op, no LLM needed

real_builder.add_edge(START, 'triage')
real_builder.add_conditional_edges('triage', route_verdict)
real_builder.add_edge('moderate', END)
real_builder.add_edge('passthrough', END)

moderation_graph = real_builder.compile()
print('Full moderation graph compiled.')
print()
print(moderation_graph.get_graph().draw_mermaid())

In [ ]:
test_messages = [
    'Check out my crypto trading course — 99% win rate! DM me now!!!',
    'Does anyone know a good Python tutorial for beginners?',
]

for msg in test_messages:
    print('=' * 60)
    print(f'Message: {msg!r}')
    print('-' * 60)

    state = {
        'messages': [HumanMessage(content=msg)],
        'verdict': '',
        'reason': '',
        'confidence': 0.0,
        'action_taken': '',
    }
    result = moderation_graph.invoke(state)

    v = result['verdict']
    r = result['reason']
    c = result['confidence']
    a = result['action_taken']
    print(f'Verdict:    {v}')
    print(f'Reason:     {r}')
    print(f'Confidence: {c:.0%}')
    print(f'Action:     {a}')
    print()

## WHAT — `create_agent` vs Manual `StateGraph`

`create_agent()` returns the exact same `CompiledStateGraph` type you just built.
The difference is the *shape* of the graph it creates — a **ReAct loop** (agent → tools → agent).

Let's call `draw_mermaid()` on a `create_agent` result to see its internal structure.

In [ ]:
from langchain.agents import create_agent


# create_agent builds a ReAct loop: agent -> (tools or END)
agent_graph = create_agent(
    llm,
    tools=[],
    system_prompt='You are a moderation assistant.',
)

print('create_agent returns:', type(agent_graph))
print()
print('Internal graph (the ReAct loop):')
print(agent_graph.get_graph().draw_mermaid())

### When to use each approach

| Your need | Tool |
|-----------|------|
| Single LLM that uses tools (Q&A bot, chat) | `create_agent()` |
| Multi-step pipeline with distinct stages | Manual `StateGraph` |
| Conditional routing based on intermediate LLM output | Manual `StateGraph` |
| Parallel branches | Manual `StateGraph` |
| Rapid prototyping | `create_agent()` |
| Memory / checkpointing | Either (same `.compile(checkpointer=...)` API) |

`create_agent()` is a factory for *one specific* `StateGraph` pattern.
When your architecture needs something different — like the moderation pipeline — you reach
for `StateGraph` directly. Same underlying machinery, more control.

## HOW — Packaging as a Factory Function

Extract everything into a `build_moderation_graph()` factory so it is importable and configurable.

In [ ]:
def build_moderation_graph(
    model: str = 'gpt-4o-mini',
    temperature: float = 0,
):
    '''Build and compile the 3-node moderation StateGraph.

    Parameters
    ----------
    model : str
        OpenAI model name used for both triage and moderation nodes.
    temperature : float
        Sampling temperature. Use 0 for deterministic classification.

    Returns
    -------
    langgraph.graph.state.CompiledStateGraph
        A compiled graph ready to call with .invoke() or .ainvoke().
    '''
    _llm = ChatOpenAI(model=model, temperature=temperature)
    _triage_llm = _llm.with_structured_output(TriageResult)
    _moderate_llm = _llm.with_structured_output(ModerationDecision)

    def _triage(state: ModerationState) -> dict:
        msg = state['messages'][-1].content if state['messages'] else ''
        result: TriageResult = _triage_llm.invoke([
            SystemMessage(content=TRIAGE_SYSTEM_PROMPT),
            HumanMessage(content=f'Message to analyze: {msg!r}'),
        ])
        return {'verdict': result.verdict, 'reason': result.reason, 'confidence': result.confidence}

    def _moderate(state: ModerationState) -> dict:
        msg = state['messages'][-1].content if state['messages'] else ''
        reason = state['reason']
        conf = state['confidence']
        user_content = f'Flagged: {msg!r} | Reason: {reason} | Confidence: {conf:.0%}'
        result: ModerationDecision = _moderate_llm.invoke([
            SystemMessage(content=MODERATION_SYSTEM_PROMPT),
            HumanMessage(content=user_content),
        ])
        return {'action_taken': f'{result.action}: {result.explanation}'}

    def _passthrough(state: ModerationState) -> dict:
        return {'action_taken': 'none'}

    def _router(state: ModerationState) -> Literal['moderate', 'passthrough']:
        return 'moderate' if state['verdict'] == 'flagged' else 'passthrough'

    b = StateGraph(ModerationState)
    b.add_node('triage', _triage)
    b.add_node('moderate', _moderate)
    b.add_node('passthrough', _passthrough)
    b.add_edge(START, 'triage')
    b.add_conditional_edges('triage', _router)
    b.add_edge('moderate', END)
    b.add_edge('passthrough', END)
    return b.compile()


# Quick smoke-test — compiles without making any API calls
graph = build_moderation_graph()
print('build_moderation_graph() works. Type:', type(graph))
print()
print(graph.get_graph().draw_mermaid())

In [ ]:
# Full end-to-end test using the factory
graph = build_moderation_graph()
msg = 'You are all idiots and deserve to be banned!!!'

result = graph.invoke({
    'messages': [HumanMessage(content=msg)],
    'verdict': '',
    'reason': '',
    'confidence': 0.0,
    'action_taken': '',
})

v = result['verdict']
r = result['reason']
c = result['confidence']
a = result['action_taken']
print(f'Message:    {msg!r}')
print(f'Verdict:    {v}')
print(f'Reason:     {r}')
print(f'Confidence: {c:.0%}')
print(f'Action:     {a}')

## What You Learned

| Concept | Key Takeaway |
|---------|-------------|
| `TypedDict` state | The shared memory of your graph — every node reads/writes through it |
| `add_messages` reducer | Appends new messages instead of replacing the list |
| Nodes | `(state) -> dict` — return only what changed |
| `add_conditional_edges` | Router function maps current state to the next node name |
| `builder.compile()` | Returns `CompiledStateGraph` — same type as `create_agent()` |
| `get_graph().draw_mermaid()` | Instantly visualize any graph structure for debugging |
| `build_moderation_graph()` | Factory pattern — encapsulates all setup for clean reuse |

## What's Next — Lesson 9: Checkpointing & Human-in-the-Loop

Right now every `graph.invoke()` is **stateless** — the graph starts fresh with no memory.

In Lesson 9 you will add **persistence** using LangGraph's built-in checkpointer:

```python
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# Invocations with the same thread_id share state across calls
result = graph.invoke(state, config={'configurable': {'thread_id': 'user_123'}})
```

You will also add a **human-in-the-loop** step: an `interrupt()` that pauses the graph and waits
for a moderator to approve or override the AI decision before taking any action.